In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

In [2]:
np.random.seed(1)

#### ML and REML Estimation

In [3]:
# Simulation parameters

n_participants = 40

memory_loads = [2, 4, 6, 8]

n_trials = 15

grand_mean = 600

participant_sd = 50

trial_sd = 25

population_slope = 18

In [4]:
# Simulating participant effects.

participant_intercepts = np.random.normal(
    loc=grand_mean,
    scale=participant_sd,
    size=n_participants
)

In [5]:
# Simulating repeated observations.

participant = []

memory_load = []

reaction_time = []

for i in range(n_participants):

    baseline = participant_intercepts[i]

    for load in memory_loads:

        rt = np.random.normal(
            loc=baseline + population_slope * load,
            scale=trial_sd,
            size=n_trials
        )

        participant.extend([i + 1] * n_trials)

        memory_load.extend([load] * n_trials)

        reaction_time.extend(rt)

data = pd.DataFrame({
    "participant": participant,
    "memory_load": memory_load,
    "reaction_time": reaction_time
})

In [6]:
# Fittin model using Maximum Likelihood.

ml_model = smf.mixedlm(
    "reaction_time ~ memory_load",
    data=data,
    groups=data["participant"]
).fit(reml=False)

In [7]:
# Fitting model using REML.

reml_model = smf.mixedlm(
    "reaction_time ~ memory_load",
    data=data,
    groups=data["participant"]
).fit(reml=True)

In [8]:
# Summary statistics

print("Maximum Likelihood (ML)\n", ml_model.summary())

print("\nRestricted Maximum Likelihood (REML)\n", reml_model.summary())

print("\nModel Comparison")

comparison = pd.DataFrame({
    "Statistic": [
        "Log-Likelihood",
        "AIC",
        "BIC",
        "Residual Variance",
        "Random Intercept Variance"
    ],
    "ML": [
        ml_model.llf,
        ml_model.aic,
        ml_model.bic,
        ml_model.scale,
        ml_model.cov_re.iloc[0, 0]
    ],
    "REML": [
        reml_model.llf,
        reml_model.aic,
        reml_model.bic,
        reml_model.scale,
        reml_model.cov_re.iloc[0, 0]
    ]
})

print(comparison)

Maximum Likelihood (ML)
            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: reaction_time
No. Observations: 2400    Method:             ML           
No. Groups:       40      Scale:              632.3478     
Min. group size:  60      Log-Likelihood:     -11253.3220  
Max. group size:  60      Converged:          Yes          
Mean group size:  60.0                                     
-----------------------------------------------------------
              Coef.   Std.Err.   z    P>|z|  [0.025  0.975]
-----------------------------------------------------------
Intercept     596.009    7.827 76.152 0.000 580.669 611.349
memory_load    18.043    0.230 78.598 0.000  17.593  18.492
Group Var    2386.992   21.498                             


Restricted Maximum Likelihood (REML)
            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: reaction_time
No. Observations: 2400    Method:             REML       